# Exp 6: Symmetric Hash Join (IVM-style Delta Processing)

Both sides (R=PART, S=LINEITEM) maintain hash tables.
Delta arrives on each side: **Δ(R ⋈ S) = (ΔR ⋈ S) ∪ (R ⋈ ΔS)**

Compares: **SNAP** (full rebuild per delta) vs **MONO-WR, DUAL-WR, EPOCH-WR** (incremental)

Three sub-experiments:
- **6a**: Stacked bar – per-round operation breakdown (4 configs)
- **6b**: Line chart – per-round latency as rounds increase (SNAP grows, MVHT flat)
- **6c**: (Future) Multi-thread scalability

**Run all cells top-to-bottom.**

In [1]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'matplotlib'])
print('done')

done



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
from pathlib import Path
import subprocess, os, sys
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np

ROOT      = Path('../../').resolve()
EXP6_DIR  = (ROOT / 'benches' / 'exp6_symmetric_join').resolve()
DATA_DIR  = EXP6_DIR / 'data'
FIGS_DIR  = EXP6_DIR / 'figs'
TPCH_DIR  = (ROOT / 'benches' / 'sigmod' / 'tpch_data').resolve()
BIN       = ROOT / 'target' / 'release' / 'symmetric_bench'

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# ===== CONFIG (aligned with Exp 3) =====
SF          = '1.0'
BUCKET_NUM  = 4096
WARMUP      = 1
REPEAT      = 10
TRIM        = 2
ROUNDS      = 5
UPDATE_PCT  = 1
DIST        = 'uniform'

# EPOCH split frequency: 0 = only initial split, no per-round splits
SPLIT_EVERY = 0

# Data files
PART_FILE       = TPCH_DIR / f'part_sf{SF}.tbl'
LINEITEM_FILE   = TPCH_DIR / f'lineitem_probe_sf{SF}_1995-09-01_1995-10-01.tbl'
UPDATES_FILE    = TPCH_DIR / f'part_updates_sf{SF}_{UPDATE_PCT}pct_{DIST}.tbl'

# Delta-S files (generated by generate_delta_s.py)
LINEITEM_INITIAL = DATA_DIR / 'lineitem_initial.tbl'
LINEITEM_DELTA   = DATA_DIR / 'lineitem_delta.tbl'

# WR-only series (justified: Exp 1-2 showed WR dominates in read-dominant workloads)
SERIES = [
    ('snap', 'nr'),   # SNAP baseline
    ('heap', 'wr'),   # MONO-WR
    ('chain','wr'),   # DUAL-WR
    ('par',  'wr'),   # EPOCH-WR
]

DISPLAY_NAMES = {
    ('snap','nr'):   'SNAP',
    ('heap','wr'):   'MONO-WR',
    ('chain','wr'):  'DUAL-WR',
    ('par','wr'):    'EPOCH-WR',
}

# Paul Tol Bright palette
tol = {
    'blue':   '#4477AA',
    'cyan':   '#66CCEE',
    'green':  '#228833',
    'yellow': '#CCBB44',
    'red':    '#EE6677',
    'purple': '#AA3377',
    'grey':   '#BBBBBB',
}

SERIES_COLORS = {
    ('snap','nr'):   tol['red'],
    ('heap','wr'):   tol['blue'],
    ('chain','wr'):  tol['yellow'],
    ('par','wr'):    tol['green'],
}

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif']  = ['Times New Roman', 'Times', 'Nimbus Roman No9 L', 'DejaVu Serif']

print('ROOT       :', ROOT)
print('PART       :', PART_FILE)
print('LINEITEM   :', LINEITEM_FILE)
print('UPDATES    :', UPDATES_FILE)
print('SPLIT_EVERY:', SPLIT_EVERY)

In [3]:
# ── Build Rust binary ──────────────────────────────────────────────────────
print('Building symmetric_bench...')
result = subprocess.run(
    ['cargo', 'build', '--release', '--bin', 'symmetric_bench'],
    cwd=ROOT, capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('cargo build failed')
print('Build OK')

Building symmetric_bench...
Build OK


In [4]:
# ── Verify data files & generate delta-S ──────────────────────────────────
for f in [PART_FILE, LINEITEM_FILE, UPDATES_FILE]:
    if not f.exists():
        raise FileNotFoundError(f'Missing: {f}')
    rows = sum(1 for _ in open(f))
    print(f'  {f.name}: {rows:,} rows')

# Generate initial + delta split for LINEITEM (S-side)
GEN_DELTA_S = EXP6_DIR / 'generate_delta_s.py'
if not LINEITEM_INITIAL.exists() or not LINEITEM_DELTA.exists():
    print('\nGenerating delta-S split...')
    r = subprocess.run(
        [sys.executable, str(GEN_DELTA_S),
         str(LINEITEM_FILE), str(DATA_DIR),
         '--initial_pct', '80', '--seed', '42'],
        check=True, capture_output=True, text=True,
    )
    print(r.stdout.strip())
else:
    print('\nDelta-S files already exist.')

for f in [LINEITEM_INITIAL, LINEITEM_DELTA]:
    rows = sum(1 for _ in open(f))
    print(f'  {f.name}: {rows:,} rows')

FileNotFoundError: Missing: /Users/jun/Desktop/dev/mbp16/crustydb/riki/FosterBtree/benches/sigmod/tpch_data/part_sf1.tbl

In [ ]:
# ── Helper: run symmetric_bench ───────────────────────────────────────────
def run_symmetric(table_type, repair_mode, rounds,
                  output_csv, per_round_csv=None,
                  split_every=SPLIT_EVERY,
                  extra_args=None):
    """Run symmetric_bench binary and return True on success."""
    cmd = [
        str(BIN),
        '--r-file',        str(PART_FILE),
        '--s-file',        str(LINEITEM_INITIAL),
        '--delta-r-file',  str(UPDATES_FILE),
        '--delta-s-file',  str(LINEITEM_DELTA),
        '--table-type',    table_type,
        '--repair-mode',   repair_mode,
        '--bucket-num',    str(BUCKET_NUM),
        '--rounds',        str(rounds),
        '--warmup',        str(WARMUP),
        '--repeat',        str(REPEAT),
        '--trim',          str(TRIM),
        '--split-every',   str(split_every),
        '--output-csv',    str(output_csv),
    ]
    if per_round_csv:
        cmd.extend(['--per-round-csv', str(per_round_csv)])
    if extra_args:
        cmd.extend(extra_args)

    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  FAILED: {r.stderr[:500]}')
        return False
    # Print summary
    for line in r.stderr.strip().split('\n'):
        if 'Average' in line or 'total_ms' in line or 'build_' in line:
            print(f'  {line.strip()}')
    return True

---
## Exp 6a: Stacked Bar – Per-Round Operation Breakdown

Single configuration comparison: average per-round cost broken into 4 phases:
- `delta_r_update`: Apply ΔR to R-side table (SNAP: full rebuild, MVHT: incremental update)
- `delta_r_probe`: Probe S-side with ΔR keys
- `delta_s_insert`: Apply ΔS to S-side table (SNAP: full rebuild, MVHT: incremental insert)
- `delta_s_probe`: Probe R-side with ΔS keys

**Expected**: SNAP's update/insert segments dominate (rebuild cost). MVHT variants are small.

In [ ]:
# ── Exp 6a: Run all configs ───────────────────────────────────────────────
CSV_6A = DATA_DIR / f'exp6a_breakdown_sf{SF}_{UPDATE_PCT}pct.csv'

if CSV_6A.exists():
    CSV_6A.unlink()
    print('Removed old CSV')

for i, (ttype, rmode) in enumerate(SERIES):
    label = DISPLAY_NAMES[(ttype, rmode)]
    print(f'[{i+1}/{len(SERIES)}] {label} (table={ttype}, repair={rmode})')
    run_symmetric(ttype, rmode, ROUNDS, CSV_6A)

print(f'\nDone. Results -> {CSV_6A}')

In [ ]:
# ── Plot 6a: Stacked bar (paper theme) ───────────────────────────────────
df6a = pd.read_csv(CSV_6A)

# Normalize table_type names (Rust Debug format: Heap, Chain, Par, Snap)
tmap = {'Heap': 'heap', 'Chain': 'chain', 'Par': 'par', 'Snap': 'snap'}
rmap = {'Nr': 'nr', 'Rr': 'rr', 'Wr': 'wr'}
df6a['table']  = df6a['table_type'].map(tmap).fillna(df6a['table_type'].str.lower())
df6a['repair'] = df6a['repair_mode'].map(rmap).fillna(df6a['repair_mode'].str.lower())

# Phase definitions: (csv_col, label, color, is_read)
# Write phases → solid fill; Read phases → white fill + colored hatch
PHASES = [
    ('avg_delta_r_update_ms', 'ΔR update/rebuild', tol['red'],    False),
    ('avg_delta_r_probe_ms',  'ΔR probe S',        tol['cyan'],   True),
    ('avg_delta_s_insert_ms', 'ΔS insert/rebuild', tol['purple'], False),
    ('avg_delta_s_probe_ms',  'ΔS probe R',        tol['green'],  True),
]

# Order bars by SERIES
bar_data = []
for (ttype, rmode) in SERIES:
    row = df6a[(df6a['table'] == ttype) & (df6a['repair'] == rmode)]
    if row.empty:
        print(f'  WARNING: no data for {ttype}/{rmode}')
        continue
    row = row.iloc[0]
    bar_data.append({
        'label': DISPLAY_NAMES[(ttype, rmode)],
        **{p[0]: float(row[p[0]]) for p in PHASES},
    })

fig, ax = plt.subplots(figsize=(6.5, 4.5))

x = np.arange(len(bar_data))
bar_w = 0.58
bottoms = np.zeros(len(bar_data))

legend_handles = []

for (col, label, color, is_read) in PHASES:
    vals = np.array([d[col] for d in bar_data])
    if is_read:
        # READ phase: white fill + colored diagonal hatch
        ax.bar(x, vals, bar_w, bottom=bottoms,
               color='white', edgecolor='black', linewidth=0.4, zorder=2)
        ax.bar(x, vals, bar_w, bottom=bottoms,
               color='none', edgecolor=color, hatch='///',
               linewidth=0.9, zorder=3)
        legend_handles.append(Patch(facecolor='white', edgecolor=color,
                                     hatch='///', linewidth=0.9, label=label))
    else:
        # WRITE phase: solid fill
        ax.bar(x, vals, bar_w, bottom=bottoms,
               color=color, edgecolor='black', linewidth=0.4, zorder=2)
        legend_handles.append(Patch(facecolor=color, edgecolor='black',
                                     linewidth=0.4, label=label))
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels([d['label'] for d in bar_data], fontsize=11)
ax.set_ylabel('Avg Per-Round Latency (ms)', fontsize=11)
ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
ax.legend(handles=legend_handles, fontsize=9, framealpha=0.9, loc='upper left')
ax.set_title(f'Symmetric Join: Per-Round Breakdown (SF={SF}, {ROUNDS} rounds)')

plt.tight_layout()
out_pdf = FIGS_DIR / f'exp6a_breakdown_sf{SF}.pdf'
plt.savefig(str(out_pdf), format='pdf')
plt.show()
print('Saved:', out_pdf)

---
## Exp 6b: Per-Round Latency Trend

Run with more rounds (e.g., 10) and plot per-round total latency.

**Expected**: SNAP's S-side rebuild cost increases each round (LINEITEM grows with each ΔS batch).
MVHT variants remain flat — incremental insert/update cost is constant per batch.

In [ ]:
# ── Exp 6b: Per-round trend ──────────────────────────────────────────────
ROUNDS_6B = 10
CSV_6B_PERROUND = DATA_DIR / f'exp6b_perround_sf{SF}_{UPDATE_PCT}pct.csv'

if CSV_6B_PERROUND.exists():
    CSV_6B_PERROUND.unlink()
    print('Removed old CSV')

# Use a dummy avg csv (we only care about per-round csv here)
CSV_6B_AVG = DATA_DIR / f'exp6b_avg_sf{SF}_{UPDATE_PCT}pct.csv'
if CSV_6B_AVG.exists():
    CSV_6B_AVG.unlink()

for i, (ttype, rmode) in enumerate(SERIES):
    label = DISPLAY_NAMES[(ttype, rmode)]
    print(f'[{i+1}/{len(SERIES)}] {label} (rounds={ROUNDS_6B})')
    run_symmetric(ttype, rmode, ROUNDS_6B, CSV_6B_AVG,
                  per_round_csv=str(CSV_6B_PERROUND))

print(f'\nDone. Per-round results -> {CSV_6B_PERROUND}')

In [ ]:
# ── Plot 6b: Per-round latency trend (paper theme) ──────────────────────
df6b = pd.read_csv(CSV_6B_PERROUND)

tmap = {'Heap': 'heap', 'Chain': 'chain', 'Par': 'par', 'Snap': 'snap'}
rmap = {'Nr': 'nr', 'Rr': 'rr', 'Wr': 'wr'}
df6b['table']  = df6b['table_type'].map(tmap).fillna(df6b['table_type'].str.lower())
df6b['repair'] = df6b['repair_mode'].map(rmap).fillna(df6b['repair_mode'].str.lower())

# Line styles match paper: SNAP=dotted, WR=solid; colors=table type
LINES_6B = [
    ('snap',  'nr', 'SNAP',     tol['red'],    ':',  'x',  8),
    ('heap',  'wr', 'MONO-WR',  tol['blue'],   '-',  'o',  6),
    ('chain', 'wr', 'DUAL-WR',  tol['yellow'], '-',  's',  6),
    ('par',   'wr', 'EPOCH-WR', tol['green'],  '-',  'D',  6),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: total round time
for (t, r, label, color, ls, marker, ms) in LINES_6B:
    rows = df6b[(df6b['table'] == t) & (df6b['repair'] == r)].sort_values('round')
    if rows.empty:
        continue
    ax1.plot(rows['round'], rows['total_round_ms'],
             label=label, color=color, linestyle=ls, marker=marker,
             linewidth=1.8, markersize=ms)

ax1.set_xlabel('Round', fontsize=11)
ax1.set_ylabel('Per-Round Latency (ms)', fontsize=11)
ax1.set_title('Total Round Latency')
ax1.set_axisbelow(True)
ax1.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
ax1.legend(fontsize=9, framealpha=0.9)

# Right: S-side insert/rebuild time (where SNAP should grow)
for (t, r, label, color, ls, marker, ms) in LINES_6B:
    rows = df6b[(df6b['table'] == t) & (df6b['repair'] == r)].sort_values('round')
    if rows.empty:
        continue
    ax2.plot(rows['round'], rows['delta_s_insert_ms'],
             label=label, color=color, linestyle=ls, marker=marker,
             linewidth=1.8, markersize=ms)

ax2.set_xlabel('Round', fontsize=11)
ax2.set_ylabel('ΔS Insert/Rebuild (ms)', fontsize=11)
ax2.set_title('S-Side Maintenance Cost')
ax2.set_axisbelow(True)
ax2.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
ax2.legend(fontsize=9, framealpha=0.9)

fig.suptitle(f'Symmetric Join: Per-Round Trend (SF={SF}, {ROUNDS_6B} rounds)', fontsize=12)
plt.tight_layout()
out_pdf = FIGS_DIR / f'exp6b_perround_sf{SF}.pdf'
plt.savefig(str(out_pdf), format='pdf')
plt.show()
print('Saved:', out_pdf)

---
## Exp 6c: Multi-Thread Scalability (Future)

TODO: Add `--reader-threads` / `--writer-threads` to `symmetric_bench.rs`.
Same pattern as exp5: writer applies deltas, readers concurrently probe the other side.

For now, the benchmark runs single-threaded.